# SVI Applied to the Results from Coaching Experiments in Eight Schools

This notebook applies SVI to data from eight schools coaching experiments, presented in chapter 5 of "Bayesian Data Analysis, 3rd edition, Gelman, Carlin, Stern, Dunson, Vehtari, and Rubin, CRC Press 2013", using MCMC (NUTS) and Stochastic Variational Inference.

***

The data was collected in a study that aimed to analyze the effects of special coaching programs on students test scores. Separate randomized experiments were performed to estimate the effects of coaching programs SAT-V in each of eight high schools.
The outcome variable in each study was the score the sudent got after administrating the SAT-V program. The scores can vary between 200 and 800, with mean about 500 and standard deviation about 100. There was no prior reason to believe that any of
the eight programs was more effective than any other or that some were more similar in effect to each other than to any.

The estimated coaching effects are label $y_j$ and their sampling variances is $\sigma_j^2$. The estimates $y_j$ are obtained by independent experiments and have approximately normal sampling distributions with sampling variances that are known, because the sample sizes in all of the eight experiments were relatively large, over thirty students in each school.

It is assumed that the parameters $\theta_j$ are drawn from a normal distribution with hyperparameters $(\mu,\tau)$:<P>

$p(\theta_1,\dots,\theta_J \mid \mu,\tau) = \prod_{j=1}^J \mathcal{N}(\theta_j \mid \mu, \tau^2)$

$p(\theta_1,\dots,\theta_J) = \int \prod_{j=1}^J \left[ \mathcal{N}(\theta_j \mid \mu, \tau^2) \right] p(\mu,\tau) d\mu d\tau$

That is, the $\theta_j$'s are conditionally independent given $(\mu, \tau)$. The assumed hierarchical model permits the interpretation that the $\theta_j$'s are random samples from a shared population distribution.

A uniform prior density for $\mu$ is reasonable for this problem because the combined data from all $J$ experiments are generally highly informative about $\mu$, and so we can be vague about its prior distribution.

Combining the sampling model for the observed effects $y_{ij}$'s and the prior distribution yields the joint posterior distribution of all the parameters and hyperparameters, which we can express in terms of the sufficient statistics $\bar{y}_{.j}$ (usually called $\eta$):

\begin{aligned}
p(\theta, \mu, \tau \mid y) & \propto p(\mu, \tau) p(\theta \mid \mu, \tau) p(y \mid \theta) \\
    & \propto p(\mu, \tau)  \prod_{j=1}^J \mathcal{N}(\theta_j \mid \mu, \tau^2) 
    \prod_{j=1}^J \mathcal{N}(\bar{y}_{.j} \mid \theta_j, \sigma_j^2)
\end{aligned}

where we can ignore factors that depend only on $y$ and the parameters $\sigma_j$, which are assumed known for this analysis.

The starting point is the following model coded in Stan.

```
data {
  int<lower=0>  J;        // number of schools (or number of coaching programs)
  real          y[J];     // estimated coaching effects
  real<lower=0> sigma[J]; // sandard error of coaching effect estimates
}

parameters {
  real          mu;
  real<lower=0> tau;
  real          eta[J]; 
}

transformed parameters {
  real theta[J];
  for (j in 1:J)
    theta[j] <- mu + tau * eta[j];
}

model {
  eta ~ normal(0, 1);
  y   ~ normal(theta, sigma);
}
```

Therefore, we can summarize our statistical probabilistic problem as follows:

* hidden local variables (or parameters):
  - $mu$ (real)
  - $tau$ (positive real)
  - $eta_1$..$eta_J$ (real)

* hidden global variables:
  - $theta_1$..$theta_J$ (real) $\rightarrow$ $theta_j = mu + tau * eta_j$

* model:
  - $eta \sim Normal(0, 1)$
  - $y   \sim Normal(theta, sigma)$

In [ ]:
import logging

import torch
from   torch.distributions import constraints, transforms

import pyro
import pyro.distributions as dist
from   pyro.infer import SVI, JitTrace_ELBO, Trace_ELBO
from   pyro.optim import Adam
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

In [ ]:
assert pyro.__version__.startswith("1.9.0")
    
logging.basicConfig(format="%(message)s", level=logging.INFO)

In [ ]:
# Number of schools or coaching programs
J     = 8

# Estimated coaching effect <=> mean
y     = torch.tensor([28, 8, -3, 7, -1, 1, 18, 12]).type(torch.Tensor)

# Standard deviation of coaching effect <=> sqrt(variance)
sigma = torch.tensor([15, 10, 16, 11, 9, 11, 10, 18]).type(torch.Tensor)

data  = torch.stack([y, sigma], dim=1)

In [ ]:
print(data)

In [ ]:
def model(sigma, y=None):
    #y     = data[:, 0]
    #sigma = data[:, 1]

    with pyro.plate("data", J):
        # mu ~ N(0,10)
        mu  = pyro.sample("mu",  dist.Normal     (torch.zeros(1), 10 * torch.ones(1)))
        
        # HalfCauchy(mu=0,sigma=25) = fat-tailed normal with non-zero PDF(y) only for y>mu.
        # tau = HalfCauchy(0,25)
        tau = pyro.sample("tau", dist.HalfCauchy (scale=25 * torch.ones(1)))

        # eta[1..J] ~ N(0,1) 
        eta = pyro.sample("eta", dist.Normal     (torch.zeros(J), torch.ones(J)))

        theta = mu + tau * eta

        # y ~ N(theta,sigma)
        pyro.sample("obs", dist.Normal(theta, sigma), obs=y)

In [ ]:
def guide(sigma, y=None):
    # Initialize the 'scales' to be quite narrow
    loc_eta      = torch.randn(J)
    scale_eta    = 0.1 * torch.rand(J)

    loc_mu       = torch.randn(1)
    scale_mu     = 0.1 * torch.rand(1)

    loc_logtau   = torch.randn(1)         # use log(tau) to be a real value an not contrained
    scale_logtau = 0.1 * torch.rand(1)

    # Register the learnable params in the parameter store
    m_eta_param    = pyro.param("loc_eta",      loc_eta)
    s_eta_param    = pyro.param("scale_eta",    scale_eta, constraint=constraints.positive)
    
    m_mu_param     = pyro.param("loc_mu",       loc_mu)
    s_mu_param     = pyro.param("scale_mu",     scale_mu, constraint=constraints.positive)

    m_logtau_param = pyro.param("loc_logtau",   loc_logtau)
    s_logtau_param = pyro.param("scale_logtau", scale_logtau, constraint=constraints.positive
    )

    # Variational distributions for eta, mu, tau
    
    dist_eta = dist.Normal(m_eta_param, s_eta_param)
    
    dist_mu  = dist.Normal(m_mu_param,  s_mu_param)
    
    # Distribution that results from transforming the Normal with an Exponential
    # The exponential reverts the effect of log(tau): e^{log(tau)} = tau
    dist_tau = dist.TransformedDistribution(
        dist.Normal(m_logtau_param, s_logtau_param),
        transforms=transforms.ExpTransform()
    )

    with pyro.plate("data", J):
        pyro.sample("eta", dist_eta)
        pyro.sample("mu",  dist_mu)
        pyro.sample("tau", dist_tau)

In [ ]:
def run_SVI(lr, epochs, jit):

    optim = Adam({"lr": lr})
    elbo  = JitTrace_ELBO() if jit else Trace_ELBO()
    svi   = SVI(model, guide, optim, loss=elbo)

    pyro.clear_param_store()

    # run SVI to optimize the ELBO
    for j in range(epochs):
        loss = svi.step(data[:, 1], data[:, 0])
        if j % 100 == 99:
            logging.info("[epoch %04d] loss: %.4f" % (j + 1, loss))

    # Print estimated variational parameters
    for name, value in pyro.get_param_store().items():
        print(f'\n{name}: ',end='')
        val = value.detach().cpu().numpy()
        if len(val) == 1:
            print(f'{val[0] :.4f}',end='')
        else:
            for v in val:
                print(f'{v :.4f}',end='  ')

In [ ]:
lr         = 0.01   # learning rate
num_epochs = 1000   # number of epochs
jit        = False  # use just in time compiler?

# Optimize the ELBO to find the best variational parameters (eta, mu, tau)

run_SVI(lr, num_epochs, jit)

In [ ]:

loc_eta      = pyro.param('loc_eta').data.cpu().numpy()
scale_eta    = pyro.param('scale_eta').data.cpu().numpy()
loc_mu       = pyro.param('loc_mu').data.cpu().numpy()
scale_mu     = pyro.param('scale_mu').data.cpu().numpy()
loc_logtau   = pyro.param('loc_logtau').data.cpu().numpy()
scale_logtau = pyro.param('scale_logtau').data.cpu().numpy()


In [ ]:
loc_theta = loc_mu + np.exp(loc_logtau) * loc_eta
print(loc_theta)

In [ ]:
predictive  = pyro.infer.Predictive(model, guide=guide, num_samples=800)
svi_samples = predictive(data[:, 1], y=None)
svi_y       = svi_samples["obs"]
svi_y       = svi_y.detach().cpu().numpy()

svi_y_mean  = svi_y.mean(axis=0)
print(f'mean: {svi_y_mean}')

In [ ]:
fig, axes = plt.subplots(8, 1, sharex='col', sharey='col')
fig.set_size_inches(6, 16)

for i in range(J): # J = #schools
  sns.kdeplot(svi_y[:][i], ax=axes[i], fill=True)
  axes[i].title.set_text(f'School {i} coaching effect distribution')

axes[J - 1].set_xlabel("School effect")
fig.tight_layout()
plt.show()

In [ ]:
# Compute the 95% interval for coaching effect (svi_y)

svi_y_low = np.array([
    np.percentile(svi_y[:, i], 2.5) for i in range(J)
])

svi_y_med = np.array([
    np.percentile(svi_y[:, i], 50) for i in range(J)
])

svi_y_hi = np.array([
    np.percentile(svi_y[:, i], 97.5) for i in range(J)
])

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, sharex=True)

ax.scatter(np.array(range(J)),       svi_y_med,       color='red',  s=60)
ax.scatter(np.array(range(J)) + 0.1, y.cpu().numpy(), color='blue', s=60)

plt.plot([-0.2, 7.4], [np.mean(svi_y), np.mean(svi_y)], 'k', linestyle='--')

ax.errorbar(
    np.array(range(J)),
    svi_y_med,
    yerr = [svi_y_med - svi_y_low, svi_y_hi  - svi_y_med],
    fmt='none'
)

ax.legend(('SVI', 'observed', 'mean effect'), fontsize=14)

plt.xlabel('School')
plt.ylabel('Coaching effect')
plt.title('SVI estimated school coaching effect vs. observed data')
fig.set_size_inches(10, 8)
plt.show()